# Book Market Analytics - Beginner Practical Notebook

**Data Science 500 | Practical work**

This notebook explains the assignment code in small parts. It collects book data, cleans it, explores it with charts, and builds a simple price prediction model. Comments explain the purpose of the main steps.

> Keep this notebook and `books_raw_scraped.csv` in the same folder. The notebook loads the provided scrape by default, so you can learn the analysis without waiting for the website. The full scraper is included below if you later want to collect the data again.

## Part 1 - Import the tools

Import the libraries we need. A library is a collection of code someone else has written so we can perform common tasks without building everything from scratch.

In [ ]:
# Read web pages and parse their HTML
from urllib.request import Request, urlopen
from urllib.parse import urljoin
from bs4 import BeautifulSoup

# Work with tables and numbers
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Make charts
import matplotlib.pyplot as plt

# Split data and build/evaluate a regression model
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Website address and a simple browser identification header
website = "https://books.toscrape.com/"
headers = {"User-Agent": "Mozilla/5.0"}

# Jupyter notebooks display charts automatically after plt.show().

## Part 2 - Get the book data

The assignment requires using BeautifulSoup. The next code cells show how to get a page, read its HTML, and collect book information from category pages. The optional live scrape visits product detail pages too, to get the description and stock quantity. It may take several minutes, so the included scrape is loaded by default.

### Part 2A - Make a helper to read a webpage

`get_soup` downloads one webpage and turns the HTML into a structure BeautifulSoup can search.

In [ ]:
def get_soup(url):
    # Ask the website for this page.
    request = Request(url, headers=headers)

    # Download the page and read the HTML text.
    with urlopen(request, timeout=30) as response:
        html_text = response.read()

    # Parse the HTML so we can select the parts we need.
    soup = BeautifulSoup(html_text, "html.parser")
    return soup

### Part 2B - Read the stock information

The page shows text such as `In stock (22 available)`. This helper returns the status and the number in parentheses.

In [ ]:
def read_stock(stock_text):
    # Remove extra spaces from the text.
    stock_text = " ".join(stock_text.split())

    if "In stock" in stock_text:
        status = "In stock"
    elif "Out of stock" in stock_text:
        status = "Out of stock"
    else:
        status = "Unknown"

    # Find a number in text like "(22 available)".
    match = re.search(r"\((\d+) available\)", stock_text, flags=re.IGNORECASE)
    if match:
        quantity = int(match.group(1))
    else:
        quantity = np.nan  # NaN means the number was not shown.

    return status, quantity

### Part 2C - Scrape the catalogue

This function visits each category and every page in that category. It collects book titles, prices, ratings, and links. Then it visits each book's detail page for the quantity and description. A `set` named `seen_links` helps prevent saving the same book twice.

In [ ]:
rating_values = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

def scrape_books():
    # Open the home page and find the category links.
    home_page = get_soup(website)
    category_links = home_page.select("ul.nav-list ul a")

    all_books = []
    seen_links = set()

    # Visit each category.
    for category_link in category_links:
        category_name = category_link.get_text(strip=True)
        category_url = urljoin(website, category_link.get("href", ""))

        # The category named "All" is not a real genre category.
        if category_name.lower() == "all":
            continue

        next_page_url = category_url

        # Keep visiting pages until there is no "next" page.
        while next_page_url:
            page = get_soup(next_page_url)
            book_cards = page.select("article.product_pod")

            for card in book_cards:
                title_link = card.select_one("h3 a")
                price_tag = card.select_one(".price_color")
                rating_tag = card.select_one(".star-rating")
                stock_tag = card.select_one(".availability")

                if title_link is None or price_tag is None:
                    continue

                book_url = urljoin(next_page_url, title_link.get("href", ""))

                # Skip the book if we have already seen its link.
                if book_url in seen_links:
                    continue
                seen_links.add(book_url)

                title = title_link.get("title", title_link.get_text(strip=True))
                price_text = price_tag.get_text(strip=True)
                price = float(re.sub(r"[^0-9.]", "", price_text))

                # The rating is stored in the star-rating HTML class.
                rating_word = "Unknown"
                if rating_tag is not None:
                    for word in rating_tag.get("class", []):
                        if word in rating_values:
                            rating_word = word

                rating_number = rating_values.get(rating_word, np.nan)

                stock_text = ""
                if stock_tag is not None:
                    stock_text = stock_tag.get_text(" ", strip=True)
                stock_status, stock_quantity = read_stock(stock_text)

                # Get the description and more accurate stock details.
                description = ""
                detail_error = ""
                try:
                    detail_page = get_soup(book_url)

                    availability = detail_page.select_one(".product_main .availability")
                    if availability is not None:
                        detail_status, detail_quantity = read_stock(
                            availability.get_text(" ", strip=True)
                        )
                        if detail_status != "Unknown":
                            stock_status = detail_status
                        # Keep the listing quantity if the detail page has no number.
                        if not pd.isna(detail_quantity):
                            stock_quantity = detail_quantity

                    description_tag = detail_page.select_one("#product_description + p")
                    if description_tag is not None:
                        description = description_tag.get_text(" ", strip=True)
                except Exception as error:
                    # Keep the listing record even if a detail page fails.
                    detail_error = str(error)[:200]

                all_books.append({
                    "title": title,
                    "price_gbp": price,
                    "rating": rating_number,
                    "rating_word": rating_word,
                    "stock_status": stock_status,
                    "stock_quantity": stock_quantity,
                    "genre": category_name,
                    "description": description,
                    "book_url": book_url,
                    "detail_error": detail_error
                })

            # Find the link to the next catalogue page.
            next_link = page.select_one("li.next a")
            if next_link is None:
                next_page_url = None
            else:
                next_page_url = urljoin(next_page_url, next_link.get("href"))

    # Turn the list of book dictionaries into a table.
    books_table = pd.DataFrame(all_books)

    # Create a simple 1/0 column for stock status.
    books_table["in_stock"] = books_table["stock_status"].map({
        "In stock": 1,
        "Out of stock": 0
    })

    return books_table

### Part 2D - Load the included scrape or collect it again

The assignment asks you to provide the scraping code. It is included above. To make learning and rerunning faster, this cell loads the provided raw data. Change `use_saved_data` to `False` to run the scraper again.

In [ ]:
use_saved_data = True
raw_file = Path("books_raw_scraped.csv")

if use_saved_data and raw_file.exists():
    # Use the data file already included with this notebook.
    raw_data = pd.read_csv(raw_file)
    print("Loaded the saved raw data.")
else:
    # Collect the data again from the website.
    raw_data = scrape_books()
    raw_data.to_csv(raw_file, index=False, encoding="utf-8-sig")
    print("Scraping finished and raw data was saved.")

print("Number of book rows:", len(raw_data))
display(raw_data.head())

## Part 3 - Clean and prepare the data

Cleaning makes the data more consistent and easier to analyze. Here we standardize text, convert numeric columns, remove duplicate books, check prices, and create word counts for text fields.

In [ ]:
# Make a working copy so the raw data stays unchanged.
books = raw_data.copy()

# Standardize text: fill missing text with blanks and remove extra spaces.
text_columns = ["title", "rating_word", "stock_status", "genre", "description", "book_url"]
for column in text_columns:
    books[column] = books[column].fillna("").astype(str)
    books[column] = books[column].str.replace(r"\s+", " ", regex=True).str.strip()

# Convert these columns to numbers. Invalid values become missing values.
number_columns = ["price_gbp", "rating", "stock_quantity"]
for column in number_columns:
    books[column] = pd.to_numeric(books[column], errors="coerce")

rows_before = len(books)

# Keep only one row for each product URL.
books = books.drop_duplicates(subset="book_url", keep="first")

# Remove rows with a missing or negative price.
books = books[books["price_gbp"].notna() & (books["price_gbp"] >= 0)].copy()

# Fill missing category names and keep ratings between 1 and 5.
books["genre"] = books["genre"].replace("", "Unknown").fillna("Unknown")
books["rating"] = books["rating"].where(books["rating"].between(1, 5))

# Create 1/0 stock values so a model can use the status as a number.
books["in_stock"] = books["stock_status"].map({"In stock": 1, "Out of stock": 0})

# Count words in titles and descriptions. Blank descriptions get a count of 0.
books["title_word_count"] = books["title"].str.split().str.len()
books["description_word_count"] = books["description"].str.split().str.len().fillna(0)

# Detail-page errors are useful to check but are not an analysis feature.
detail_errors = books["detail_error"].fillna("").ne("").sum()
books = books.drop(columns=["detail_error"], errors="ignore")

print("Rows before cleaning:", rows_before)
print("Rows after cleaning:", len(books))
print("Detail-page errors:", detail_errors)
print("Books with descriptions:", books["description"].str.len().gt(0).sum())
print("Books with stock quantities:", books["stock_quantity"].notna().sum())
display(books.head())

### Part 3A - Check that cleaning worked

These checks help catch data problems before analysis. A duplicate URL count of zero means each book appears only once.

In [ ]:
print("Missing values in each column:")
print(books.isna().sum())

print("\nDuplicate book URLs:", books["book_url"].duplicated().sum())

print("\nPrice, rating, and quantity summary:")
print(books[["price_gbp", "rating", "stock_quantity"]].describe().round(2))

### Part 3B - Save the cleaned data

This is the final dataset to submit with the assignment.

In [ ]:
books.to_csv("books_cleaned.csv", index=False, encoding="utf-8-sig")
print("Saved the cleaned data as books_cleaned.csv")

## Part 4 - Explore the data with charts

Exploratory data analysis means looking at summaries and charts before fitting a model. It helps us understand the data and find patterns or unusual values.

In [ ]:
print("Number of books:", len(books))
print("Number of categories:", books["genre"].nunique())
print("Average price: GBP", round(books["price_gbp"].mean(), 2))
print("Median price: GBP", round(books["price_gbp"].median(), 2))
print("Lowest price: GBP", round(books["price_gbp"].min(), 2))
print("Highest price: GBP", round(books["price_gbp"].max(), 2))

# Correlation is a number between -1 and 1 that summarizes a linear relationship.
rating_price_correlation = books["rating"].corr(books["price_gbp"])
print("Rating-price correlation:", round(rating_price_correlation, 3))

### Part 4A - Price distribution

A histogram groups prices into ranges. Taller bars show price ranges with more books.

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(books["price_gbp"], bins=25, color="steelblue", edgecolor="white")
plt.title("Distribution of Book Prices")
plt.xlabel("Price (GBP)")
plt.ylabel("Number of books")
plt.tight_layout()
plt.show()

### Part 4B - Average price by rating

This bar chart compares the average listed price for each star rating.

In [ ]:
rating_order = ["One", "Two", "Three", "Four", "Five"]
average_by_rating = books.groupby("rating_word")["price_gbp"].mean()
average_by_rating = average_by_rating.reindex(rating_order)

plt.figure(figsize=(8, 5))
plt.bar(average_by_rating.index, average_by_rating.values, color="steelblue")
plt.title("Average Book Price by Star Rating")
plt.xlabel("Star rating")
plt.ylabel("Average price (GBP)")
plt.tight_layout()
plt.show()

### Part 4C - Price in the largest categories

There are many site categories, so this chart shows the ten categories with the most books. The average is descriptive; it does not show that category causes a price difference.

In [ ]:
books_per_category = books["genre"].value_counts().head(10).index
largest_categories = books[books["genre"].isin(books_per_category)]
average_by_category = largest_categories.groupby("genre")["price_gbp"].mean().sort_values()

plt.figure(figsize=(9, 6))
plt.barh(average_by_category.index, average_by_category.values, color="steelblue")
plt.title("Average Price in the Ten Largest Categories")
plt.xlabel("Average price (GBP)")
plt.ylabel("Site category")
plt.tight_layout()
plt.show()

### Part 4D - Rating and price relationship

Each dot is one book. The chart lets us see whether prices tend to rise or fall as ratings increase.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(books["rating"], books["price_gbp"], alpha=0.5, color="steelblue")
plt.title("Star Rating and Listed Price")
plt.xlabel("Rating from 1 to 5")
plt.ylabel("Price (GBP)")
plt.tight_layout()
plt.show()

print("Correlation:", round(rating_price_correlation, 3))

### What the charts suggest

Prices range from GBP 10.00 to GBP 59.99. The rating-price correlation is about 0.028, which is close to zero in this dataset. The ratings and prices on the practice website were randomly assigned, so these charts demonstrate analysis methods but do not describe a real book market.

## Part 5 - Build a simple price prediction model

Linear regression learns a relationship between input information (features) and a number we want to predict (the target). Here the target is price. The features are rating, stock quantity, stock status, title length, description length, and category.

In [ ]:
# y is the value we want the model to predict.
y = books["price_gbp"]

# Start with the numeric information we want to use as predictors.
numeric_features = books[[
    "rating",
    "stock_quantity",
    "in_stock",
    "title_word_count",
    "description_word_count"
]].copy()

# Fill missing numbers with the median of that column.
numeric_features = numeric_features.fillna(numeric_features.median())

# A computer model needs categories converted to numbers.
# get_dummies creates one 0/1 column for each category.
category_features = pd.get_dummies(
    books["genre"],
    prefix="genre",
    drop_first=True,
    dtype=int
)

# Put numeric and category features together into one table called X.
X = pd.concat([numeric_features, category_features], axis=1)

print("Number of rows and features:", X.shape)

### Part 5A - Split, train, and test the model

We keep some books aside as a test set. The model learns from the training set, then we check how well it predicts prices for books it did not train on.

In [ ]:
# Use 80% of the books to train and 20% to test.
# random_state=42 makes the split repeatable each time you run the notebook.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Create the linear regression model and let it learn from training data.
price_model = LinearRegression()
price_model.fit(X_train, y_train)

# Ask the trained model to predict prices for the test books.
predicted_prices = price_model.predict(X_test)

print("Training books:", len(X_train))
print("Test books:", len(X_test))

### Part 5B - Measure prediction errors

MAE and RMSE measure typical prediction error in pounds; lower is better. R-squared summarizes how much of the variation the model explains; higher is better. The baseline always predicts the average training price, giving us a simple comparison.

In [ ]:
mae = mean_absolute_error(y_test, predicted_prices)
rmse = np.sqrt(mean_squared_error(y_test, predicted_prices))
r_squared = r2_score(y_test, predicted_prices)

# Baseline: predict the training-set average for every test book.
baseline_predictions = np.repeat(y_train.mean(), len(y_test))
baseline_mae = mean_absolute_error(y_test, baseline_predictions)

print("Model MAE (GBP):", round(mae, 2))
print("Model RMSE (GBP):", round(rmse, 2))
print("Model R-squared:", round(r_squared, 3))
print("Baseline MAE (GBP):", round(baseline_mae, 2))

### Part 5C - Compare predicted and actual prices

If predictions were perfect, the dots would sit on the dashed diagonal line. Points far from the line are predictions with larger errors.

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, predicted_prices, alpha=0.6, color="steelblue")

# Draw a diagonal line as a guide for perfect predictions.
lowest_price = min(y_test.min(), predicted_prices.min())
highest_price = max(y_test.max(), predicted_prices.max())
plt.plot(
    [lowest_price, highest_price],
    [lowest_price, highest_price],
    linestyle="--",
    color="darkred"
)

plt.title("Actual Prices Compared with Model Predictions")
plt.xlabel("Actual price (GBP)")
plt.ylabel("Predicted price (GBP)")
plt.tight_layout()
plt.show()

## Part 6 - Explain the result

In the completed analysis, the model's MAE was about GBP 12.59 and the simple baseline's MAE was about GBP 12.30. Because the model has a larger MAE than the baseline, it did not improve on simply predicting the average price. The R-squared was negative, which also indicates weak performance on the held-out test data.

The assignment asks you to interpret results, not just show code. Explain that this model should not be used for real pricing decisions. The practice website states that its prices and ratings are randomly assigned, and all books in this snapshot are marked in stock.

## Part 7 - What to submit

- **Practical notebook:** this file shows code and explanations from scraping through modelling.
- **Final dataset:** submit `books_cleaned.csv`.
- **Written report:** submit the separate report PDF with your name, student number, signature, and date completed.

The raw `books_raw_scraped.csv` is also included so the notebook can be rerun without downloading all pages again.